# Line distances

In [1]:
import geopandas as gpd
import pandas as pd
import os
from shapely.geometry import Point
import numpy as np
import networkx as nx
from pyproj import Geod

## Read data

In [2]:
path = os.getcwd()
all_lines = gpd.read_feather(os.path.join(path, 'results', 'line_data.feather')).to_crs('EPSG: 4326').drop(['CDUID_1', 'CDUID_2', 'province_1', 'province_2', 'geometry'],axis=1)
cluster_data = gpd.read_feather(os.path.join(path, 'results', 'clustered_zone_data.feather')).to_crs('EPSG: 4326')
zone_data = gpd.read_feather(os.path.join(path, 'data', 'processed_data', 'zone_data.feather')).to_crs('EPSG: 4326')
node_lookup = gpd.read_feather(os.path.join(path, 'results', 'node_data.feather')).to_crs('EPSG: 4326')

In [3]:
# Create two DataFrames from the pairs
nodes_1 = all_lines[['node_1', 'geometry_1', 'cluster_1', 'voltage']].rename(columns={'node_1': 'node', 'geometry_1': 'geometry', 'cluster_1':'cluster'})
nodes_2 = all_lines[['node_2', 'geometry_2', 'cluster_2', 'voltage']].rename(columns={'node_2': 'node', 'geometry_2': 'geometry', 'cluster_2':'cluster'})
# Concatenate and set index
nodes = pd.concat([nodes_1, nodes_2], ignore_index=True).set_index('node')
nodes = gpd.GeoDataFrame(nodes, crs='EPSG: 4326')
nodes = nodes.drop_duplicates()

#### CENTROID OF CENSUS AREA
rep_points_census = zone_data.copy()
rep_points_census['geometry'] = rep_points_census.representative_point()
rep_points_census = gpd.sjoin(rep_points_census, cluster_data[['geometry']], how='left', predicate='within')
rep_points_census = rep_points_census.drop(['DGUID', 'CSDNAME', 'CDTYPE', 'LANDAREA', 'PRUID', 'province', 'Connected'],axis=1)

manual_nodes = {'NB_West':'NB_PLPR_GSS',
                'NB_East':'NB_MRMK_TSS',
                'ON_Northwest': 'ON_TBP_JCT',
                'ON_East': 'ON_ESA_TSS',
                'ON_North': 'ON_SUD_JCT',
                'QC_Central': 'QC_BRG_SWS',
                'SK': 'SK_REG_SWS'}

node_lookup = node_lookup.to_crs('EPSG:3347')

In [4]:
def find_cluster_center(rep_points, nodes, cluster):
    # Point within cluster
    lat = rep_points.geometry.y
    lon = rep_points.geometry.x
    rep_points = rep_points.drop(['geometry', 'total_capacity'], axis=1)
    rep_points = rep_points.divide(rep_points.sum(), axis=1).fillna(0)
    
    # Equal weighting for population and generation (generation split)
    rep_points['weightings'] = (0.5 * rep_points.population 
    + 0.1 * rep_points.hydro_capacity
    + 0.1 * rep_points.solar_capacity  + 0.1 * rep_points.wind_capacity
    + 0.1 * rep_points.thermal_capacity + 0.1 * rep_points.nuclear_capacity
    )

    # Calculate the weighted lat lon points, incorporating spherical coords
    ### CREDIT - CHATGPT
    # Lat Lon to cartesian
    lat_radians = np.radians(lat)
    lon_radians = np.radians(lon)
    x = np.cos(lat_radians) * np.cos(lon_radians)
    y = np.cos(lat_radians) * np.sin(lon_radians)
    z = np.sin(lat_radians)

    # Apply weights
    x = np.average(x, weights=rep_points['weightings'])
    y = np.average(y, weights=rep_points['weightings'])
    z = np.average(z, weights=rep_points['weightings'])

    # Cartesian to lat lon
    hyp = np.sqrt(x**2 + y**2)
    lat = np.degrees(np.arctan2(z, hyp))
    lon = np.degrees(np.arctan2(y, x))
    center_point = gpd.GeoSeries([Point(lon, lat)], crs='EPSG:4326')  # Point(lon, lat)

    # Find highest voltage nodes within cluster area
    cluster_nodes = nodes[nodes.cluster == cluster]
    cluster_nodes = cluster_nodes[cluster_nodes.voltage == cluster_nodes.voltage.max()]
    crs = 'EPSG:3347'
    cluster_nodes = cluster_nodes.to_crs(crs)
    center_point = center_point.to_crs(crs).iloc[0]

    distances = cluster_nodes.geometry.distance(center_point)
    node_name = distances.idxmin()

    if len(distances.index) == 1:
         closest_node = Point(cluster_nodes.geometry.x, cluster_nodes.geometry.y)
    else:
        closest_node = cluster_nodes.loc[distances.idxmin()]
        if isinstance(closest_node, gpd.GeoDataFrame):
            closest_node = closest_node.iloc[0]
            closest_node = Point(closest_node.geometry.x, closest_node.geometry.y)
        else:
            closest_node = closest_node.geometry
    return closest_node, node_name

In [5]:
# Finding centroid of each cluster
nearest_nodes = []
for cluster in cluster_data.index:
    if cluster in manual_nodes.keys():
        node_name = manual_nodes[cluster]
        closest_node = node_lookup.loc[node_name, 'geometry']
    else:
        rep_points_cluster = rep_points_census[rep_points_census.cluster == cluster].drop('cluster',axis=1)
        closest_node, node_name = find_cluster_center(rep_points_cluster, nodes, cluster)
    nearest_nodes.append({'cluster': cluster, 'node':node_name, 'geometry':closest_node})

nearest_nodes = gpd.GeoDataFrame(nearest_nodes, crs='EPSG:3347').to_crs('EPSG:4326').set_index('cluster')
os.makedirs(os.path.join(path, 'results', 'visualization_data'), exist_ok=True)
nearest_nodes.to_feather(os.path.join(path, 'results', 'visualization_data', 'nearest_nodes.feather'))

nearest_nodes['lon'] = nearest_nodes.geometry.x
nearest_nodes['lat'] = nearest_nodes.geometry.y
nearest_nodes.drop('geometry', axis=1).to_csv(os.path.join(path, 'results', 'nearest_nodes.csv'))

In [6]:
def calc_distances(edges, main_nodes):
    main_nodes = main_nodes.reset_index()
    # Dict of node distances
    results = []

    for i in range(0, len(main_nodes)):
        start_node = main_nodes.iloc[i]
        start_node_code = start_node.node
        start_cluster = start_node.cluster
        for j in range(i+1, len(main_nodes)):
            end_node = main_nodes.iloc[j]
            end_node_code = end_node.node
            end_cluster = end_node.cluster
            clusters = [start_cluster, end_cluster]

            # Create graph, only nodes inside start and end clusters
            edge_data = edges[(edges.cluster_1.isin(clusters)) | (edges.cluster_2.isin(clusters))]
            G = nx.from_pandas_edgelist(edge_data, source='node_1', target='node_2', edge_attr='line_segment_length_km')
            ## Calculate straight line distance
            geod = Geod(ellps="WGS84")
            points = main_nodes.set_index('cluster', drop=True)
            point_1 = points.loc[start_cluster, 'geometry']
            point_2 = points.loc[end_cluster, 'geometry']
            _, _, straight_distance = geod.inv(point_1.x, point_1.y, point_2.x, point_2.y)
            straight_distance /= 1000
            ## Calcualte Dijkstra Distance
            # Check if clusters are reachable
            if nx.has_path(G, start_node_code, end_node_code):
                path_distance = nx.dijkstra_path_length(G, source=start_node_code, target=end_node_code, weight='line_segment_length_km')
                shortest_path = nx.dijkstra_path(G, source=start_node_code, target=end_node_code, weight='line_segment_length_km')
            else:
                if straight_distance >= 1000:
                    continue
                else:
                    path_distance = np.inf
            results.append({'start':start_cluster,'end':end_cluster,'dijkstra_distance':path_distance,'straight_distance':straight_distance})
    return results

### Calculate Dijkstra distance

distances = pd.DataFrame(calc_distances(all_lines[~all_lines.line_segment_length_km.isna()], nearest_nodes))

### Formatting and saving
The results are saved to a csv, although this requires manual entry of the forward and backward capacities. This is due to the complex nature of the interconnections.

In [7]:
CUSTOM = False
if CUSTOM:
    distances['distance'] = None
    distances['capacity_fwd'] = None
    distances['capacity_bck'] = None
else:
    distances = pd.read_csv(os.path.join(path, 'data', 'cluster_interfaces_default.csv'))
distances.to_csv(os.path.join(path, 'results', 'cluster_interfaces.csv'), index=False)